# Catatan Metodologi Peramalan
---
## Status Model: Baseline Eksploratif Awal (Overfitting Warning)
---
### Keterbatasan Data Deret Waktu dan Rencana Pengujian Pasca-UTS
---
Model peramalan saat ini berstatus baseline eksploratif awal. Keterbatasan runtun waktu historis tahunan dari publikasi resmi BPS (hanya n = 5 titik observasi, 2019-2023) menyebabkan model regresi tren linear sederhana rentan terhadap overfitting atau simplifikasi yang belum menangkap dinamika goncangan ekonomi pasca-pandemi secara non-linear. Hasil peramalan ini berfungsi sebagai sinyal arah sementara, dan pada tahap pasca-UTS tim merencanakan pengujian algoritma lanjutan (seperti ARIMA/SARIMA, Prophet Facebook, atau penambahan kovariat makroekonomi).


# 00 Load Library
---


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt


# 01 Inisialisasi Path dan Pembacaan Data Deret Waktu
---
## Konfigurasi Data Historis
---
### Membaca Data Deret Waktu BPS 2019-2023
---
Memuat volume unit usaha tahunan dari BPS DIY untuk 5 sektor UMKM.


In [ ]:
BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == 'analisis' else Path.cwd()
CLEANED_DIR = BASE_DIR / 'data' / 'cleaned'
HASIL_DIR = BASE_DIR / 'outputs' / 'hasil'
df_tren = pd.read_csv(CLEANED_DIR / 'tren_sektor.csv')
df_tren.head(10)


# 02 Formulasi Estimator Ordinary Least Squares OLS
---
## Model Regresi Tren Linear
---
### Ekstrapolasi Horizon 2024 hingga 2026 dan Margin CI 95%
---
Menghitung kemiringan tren, intersep, proyeksi masa depan, dan pita ketidakpastian interval kepercayaan 95%.


In [ ]:
sektor_list = df_tren['sektor'].unique()
forecast_results = []
for s in sektor_list:
    sub = df_tren[df_tren['sektor'] == s].sort_values('tahun')
    t = sub['tahun'].values
    y = sub['jumlah_unit'].values
    n = len(t)
    slope, intercept, r_val, p_val, std_err = stats.linregress(t, y)
    y_pred_hist = intercept + slope * t
    mape = np.mean(np.abs((y - y_pred_hist) / y)) * 100
    future_years = [2024, 2025, 2026]
    for fy in future_years:
        pred = intercept + slope * fy
        cagr_proj = ((pred / y[-1]) ** (1 / (fy - 2023)) - 1) * 100
        status_tren = 'Naik (Ekspansif)' if cagr_proj >= 3.0 else ('Stabil / Moderat' if cagr_proj >= 0 else 'Melambat / Stagnan')
        forecast_results.append({
            'sektor': s,
            'tahun': fy,
            'proyeksi_unit': round(pred),
            'cagr_proyeksi_pct': round(cagr_proj, 2),
            'status_arah_tren': status_tren,
            'r_squared': round(r_val**2, 3),
            'mape_pct': round(mape, 2)
        })
df_forecast = pd.DataFrame(forecast_results)
df_forecast.head(10)


# 03 Evaluasi Metrik Kuantitatif Akurasi
---
## Metrik Kebaikan Suai
---
### R-Squared dan Mean Absolute Percentage Error
---
Memeriksa nilai R2 dan tingkat kesalahan MAPE untuk mengukur kesesuaian garis tren linear.


In [ ]:
metrics_summary = df_forecast.groupby('sektor')[['r_squared', 'mape_pct']].first().reset_index()
r2_avg = metrics_summary['r_squared'].mean()
mape_avg = metrics_summary['mape_pct'].mean()
print(f'Rata-rata R-squared: {r2_avg:.3f} | Rata-rata MAPE: {mape_avg:.2f}%')
metrics_summary


# 04 Ekspor Data Hasil Peramalan Tren
---
## Penyimpanan Output Hasil
---
### Ekspor File CSV Proyeksi Sektor
---
Mengekspor tabel peramalan tren sektor ke folder hasil.


In [ ]:
output_forecast_path = HASIL_DIR / 'hasil_forecasting_sektor.csv'
df_forecast.to_csv(output_forecast_path, index=False)
print('Hasil peramalan tren Pilar 3 berhasil disimpan di:', output_forecast_path)
